In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score

from scipy.stats import beta, gamma, uniform, randint

from xgboost import XGBClassifier
from boruta import BorutaPy

import importlib
import datifier
from datifier import get_data

from new_data import give_data
from utils.data_processing import load_data, standarize, variance_threshold, correlation_filter, select_from_model


# Tomek

In [7]:
X, Y, XfinalTest = load_data()

In [ ]:
X, Y, XfinalTest = load_data()
# X, XfinalTest = standarize(X, XfinalTest)
# X, XfinalTest = variance_threshold(X, XfinalTest, 0.02)
# X, XfinalTest = correlation_filter(X, XfinalTest, 0.75)
# # X, XfinalTest = select_from_model(X, Y, XfinalTest, threshold="1.7*mean")
# # X, XfinalTest = select_k_best(X, Y, XfinalTest, k=25)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, Y, test_size=0.2, random_state=42
# )

# X.shape

Xnowe, top20 = get_data()

--- STARTING STRICT PIPELINE ---
Initial features: 500
Stage 1 (Variance & Collinearity) survivors: 485
Stage 2 (L1 Penalty) survivors: 280
check
Stage 3 (RFECV) survivors: 16
Stage 4 (Shadow Feature Protocol) survivors: 10
Stage 5 (OOF Permutation) Final Selection: 10 features.
--- PIPELINE COMPLETE ---



In [19]:
Xnew = np.load("xtrain.npy")

In [21]:
Xnew = pd.DataFrame(Xnew)

In [25]:
xgb = XGBClassifier()
xgb.fit(Xnowe, Y)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [38]:
test = xgb.predict_proba(Xnowe)[:,1]

In [42]:
np.where(test > 0.5)

(array([   1,    5,    7, ..., 4996, 4997, 4998], shape=(2443,)),)

In [3]:
def xgb_custom_score(estimator, X, y):

    no_variables = X.shape[1]
    score = 0

    for th in range(5, 50, 5):
        thr = th/100
        y_probs = estimator.predict_proba(X)[:, 1]
        indices_above_threshold = np.where(y_probs > thr)[0]
        ypred_org = y_probs[indices_above_threshold]
        sorted_ind = indices_above_threshold[np.argsort(ypred_org)[::-1]]
        top_ind = sorted_ind[:1000]
        y_pred = np.zeros(len(y_probs), dtype=int)
        y_pred[top_ind] = 1
        
        y = np.array(y).ravel()
        
        tp = np.sum((y == 1) & (y_pred == 1))
        fp = np.sum((y == 0) & (y_pred == 1))

        sc = (tp * 10) - (fp * 5) - (no_variables * 200)

        if sc > score:
            score = sc
            thresh = thr

    print(thresh)
    return score

In [84]:
xgb = XGBClassifier()

xgb.fit(X, Y)

sel = SelectFromModel(xgb, threshold="mean", prefit=True, max_features=5)

X_smol = sel.transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_smol, Y, test_size=0.2, random_state=42
)

/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


In [24]:
# def custom_score(estimator, X, y):

#     y_pred = estimator.predict(X)
#     y = np.array(y).ravel()
    
#     tp = np.sum((y == 1) & (y_pred == 1))
#     fp = np.sum((y == 0) & (y_pred == 1))

#     # used_features = np.unique(
#     #     estimator.tree_.feature[
#     #         estimator.tree_.feature >= 0
#     #     ]
#     # )

#     # no_variables = len(used_features)
#     no_variables = X.shape[1]

#     return (tp * 10) - (fp * 5) - (no_variables * 200)

# grid = GridSearchCV(
#     DecisionTreeClassifier(),
#     {
#         "max_depth": [2, 5, 10, 20],
#         "min_samples_split": [2, 5, 10],
#         "min_samples_leaf": [1, 2, 4],
#         "max_features": [None, 10, 20, 50]
#     },LogisticRegression(C=1, l1_ratio=0.7, solver="saga")
#     scoring=custom_score,
#     cv=5
# )

# grid.fit(X_train, y_train)

# print("Best params:", grid.best_params_)
# print("Best score:", grid.best_score_)

KeyboardInterrupt: 

In [ ]:
# bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')
# # fit model
# bst.fit(X_train, y_train)
# # make predictions
# preds = bst.predict(X_test)

# grid = GridSearchCV(
#     XGBClassifier(),
#     {
#         "n_estimators": [5, 10, 30, 50],
#         "max_depth": [1, 5, 10],
#         "learning_rate": [0.1, 0.5, 1.0],
#         "gamma": [0, 1, 5],
#         "alpha": [0, 5, 10]
#     },
#     scoring=xgb_custom_score,
#     cv=10
# )

# grid.fit(X_train, y_train)

# print("Best params:", grid.best_params_)
# print("Best score:", grid.best_score_)

Exception ignored while calling ctypes callback function <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x79e9bed720a0>>:
Traceback (most recent call last):
  File "/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/xgboost/core.py", line 607, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 


In [ ]:
xgb_custom_score(grid.best_estimator_, X_test, y_test)

np.int64(-4815)

In [8]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import expon, norm, poisson, uniform, randint, gamma, beta

In [30]:
rcv_xgb = RandomizedSearchCV(
    XGBClassifier(),
    {"alpha": expon(scale=1),
     "gamma": expon(scale=1),
     "learning_rate": uniform(0.01, 1.0),
     "n_estimators": randint(10, 300),
     "max_depth": randint(1, 10)},
    n_iter=500,
    scoring=xgb_custom_score,
    n_jobs=-1,
    cv=10)

rcv_xgb.fit(X_train, y_train)

print("Best params:", rcv_xgb.best_params_)
print("Best score:", rcv_xgb.best_score_)

Best params: {'alpha': np.float64(0.8767248087676504), 'gamma': np.float64(2.2926267082994474), 'learning_rate': np.float64(0.5875929738564502), 'max_depth': 1, 'n_estimators': 14}
Best score: -203.0


In [31]:
xgb_custom_score(rcv_xgb.best_estimator_, X_test, y_test)

np.int64(735)

In [80]:
model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    )
model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [81]:
xgb_custom_score(model, X_test, y_test)

np.int64(825)

In [21]:
test_probs = model.predict_proba(X_test)[:, 1]

# Select top BEST_N_CONTACTS customers by predicted probability
selected_indices = np.argsort(test_probs)[::-1][:800]
selected_indices_sorted = np.sort(selected_indices)
print(f"Final customers selected: {len(selected_indices_sorted)} (top {800} by probability)")

Final customers selected: 800 (top 800 by probability)


In [54]:
best = 0

output = {}

for k in range(2, 11):
    print("="*50)
    print(f"features number = {k-1}")

    X = Xnew.iloc[:, 1:k]

    # xgb = XGBClassifier()

    # xgb.fit(X, Y)

    # sel = SelectFromModel(xgb, threshold="mean", prefit=True, max_features=k)

    # X_smol = sel.transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=0.2
    )   

    rcv_xgb = RandomizedSearchCV(
    XGBClassifier(),
    {"gamma": gamma(2,2),
     "alpha": gamma(2,2),
     "learning_rate": uniform(0.001, 0.005),
     "n_estimators": randint(25, 400),
     "max_depth": randint(1, 50),
     "tree_method":["exact"]
     },
    n_iter=50,
    scoring="neg_log_loss",
    n_jobs=-1,
    cv=5)

    rcv_xgb.fit(X_train, y_train)

    print("Best params:", rcv_xgb.best_params_)
    print("Best score:", rcv_xgb.best_score_)

    acc = accuracy_score(y_test, rcv_xgb.best_estimator_.predict(X_test))
    print(acc)

    sc = xgb_custom_score(rcv_xgb.best_estimator_, X_test, y_test)
    print(sc)

    posmax = y_test.to_numpy().sum() * 10 - X_test.shape[1]*200
    print(f"possible max: {posmax}.")

    if sc > best:
        best = sc
        output[sc] = rcv_xgb.best_estimator_

features number = 1
Best params: {'alpha': np.float64(2.1703818869519433), 'gamma': np.float64(2.7693480030063236), 'learning_rate': np.float64(0.005016676663341347), 'max_depth': 29, 'n_estimators': 393, 'tree_method': 'exact'}
Best score: -0.6834392966288751
0.554
0.05
2150
possible max: 4700.
features number = 2
Best params: {'alpha': np.float64(2.3477264149621657), 'gamma': np.float64(3.719983029832372), 'learning_rate': np.float64(0.004886472019551677), 'max_depth': 6, 'n_estimators': 287, 'tree_method': 'exact'}
Best score: -0.6769923109658438
0.573
0.05
2085
possible max: 4590.
features number = 3
Best params: {'alpha': np.float64(3.5786794495047314), 'gamma': np.float64(4.322790822297689), 'learning_rate': np.float64(0.005837990585871136), 'max_depth': 37, 'n_estimators': 333, 'tree_method': 'exact'}
Best score: -0.6666945458671365
0.601
0.35
1405
possible max: 4060.
features number = 4


KeyboardInterrupt: 

In [46]:
def custom_score(y, y_probs, no_variables):

    # no_variables = X.shape[1]
    score = 0

    for th in range(5, 50, 5):
        thr = th/100
        # y_probs = estimator.predict_proba(X)[:, 1]
        indices_above_threshold = np.where(y_probs > thr)[0]
        ypred_org = y_probs[indices_above_threshold]
        sorted_ind = indices_above_threshold[np.argsort(ypred_org)[::-1]]
        top_ind = sorted_ind[:1000]
        y_pred = np.zeros(len(y_probs), dtype=int)
        y_pred[top_ind] = 1
        
        y = np.array(y).ravel()
        
        tp = np.sum((y == 1) & (y_pred == 1))
        fp = np.sum((y == 0) & (y_pred == 1))

        sc = (tp * 10) - (fp * 5) - (no_variables * 200)

        if sc > score:
            score = sc
            thresh = thr

        posmax = y.sum() * 10 - no_variables * 200

        print(f"possible max: {posmax}")

        acc = accuracy_score(y, y_pred)
        print(f"accuracy {acc}")

    print(thresh)
    return score

In [47]:
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
history = []

print(f"--- Launching Nested Forward Search for xgb ---")

max_features = min(Xnew.shape[1], 20)
y_train = Y

for k in range(2, max_features + 1):
    X_subset = Xnew.iloc[:, 1:k]
    oof_probs = np.zeros(len(Xnew))
    
    # Outer Cross-Validation Loop
    for train_idx, val_idx in outer_cv.split(X_subset, y_train):
        X_tr, y_tr = X_subset.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_subset.iloc[val_idx], y_train.iloc[val_idx]
        
        # Isolated Inner Cross-Validation configuration (prevents data leakage)
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        
        # --- MODEL 1: XGBOOST WITH NESTED TUNING ---
        base_estimator = XGBClassifier(eval_metric='logloss', random_state=42)
        param_dist = {
            'n_estimators': randint(50, 130),
            'max_depth': randint(3, 5),
            'learning_rate': uniform(0.02, 0.07)
        }
        search = RandomizedSearchCV(base_estimator, param_dist, n_iter=10, scoring='neg_log_loss', cv=inner_cv, random_state=42, n_jobs=-1)
        search.fit(X_tr, y_tr.to_numpy().ravel())
        best_model = search.best_estimator_
        
        calibrated_clf = CalibratedClassifierCV(best_model, method='sigmoid', cv=10)
        calibrated_clf.fit(X_tr, y_tr.to_numpy().ravel())
        oof_probs[val_idx] = calibrated_clf.predict_proba(X_val)[:, 1]

    # Invoke score validation globally on the cross-validated probabilities array
    perf_metrics = custom_score(Y, oof_probs, k)
    history.append(perf_metrics)

    print(f"Vars Count: {k:2d} | Net Profit: EUR {perf_metrics:5d} | ")


--- Launching Nested Forward Search for xgb ---
possible max: 24480
accuracy 0.5504
possible max: 24480
accuracy 0.5504
possible max: 24480
accuracy 0.5504
possible max: 24480
accuracy 0.5504
possible max: 24480
accuracy 0.5504
possible max: 24480
accuracy 0.5504
possible max: 24480
accuracy 0.5504
possible max: 24480
accuracy 0.5504
possible max: 24480
accuracy 0.55
0.05
Vars Count:  2 | Net Profit: EUR  3900 | 
possible max: 24280
accuracy 0.5636
possible max: 24280
accuracy 0.5636
possible max: 24280
accuracy 0.5636
possible max: 24280
accuracy 0.5636
possible max: 24280
accuracy 0.5636
possible max: 24280
accuracy 0.5636
possible max: 24280
accuracy 0.5636
possible max: 24280
accuracy 0.5636
possible max: 24280
accuracy 0.5636
0.05
Vars Count:  3 | Net Profit: EUR  4195 | 


KeyboardInterrupt: 

In [42]:
y_test

,y
2336,0
226,1
1828,1
3845,1
3290,1
...,...
3530,0
3136,1
1103,0
635,0


In [36]:
best = 0

output = {}

for k in range(1,2):
    print("="*50)
    print(f"features number = {k}")
    xgb = XGBClassifier()

    xgb.fit(X, Y)

    sel = SelectFromModel(xgb, threshold="mean", prefit=True, max_features=k)

    X_smol = sel.transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_smol, Y, test_size=0.2
    )   

    rcv_xgb = RandomizedSearchCV(
    XGBClassifier(),
    {"alpha": expon(scale=1),
     "gamma": expon(scale=1),
     "learning_rate": uniform(0.01, 1.0),
     "n_estimators": randint(1, 400),
     "max_depth": randint(1, 10)},
    n_iter=1000,
    scoring=xgb_custom_score,
    n_jobs=-1,
    cv=10)

    rcv_xgb.fit(X_train, y_train)

    print("Best params:", rcv_xgb.best_params_)
    print("Best score:", rcv_xgb.best_score_)

    sc = xgb_custom_score(rcv_xgb.best_estimator_, X_test, y_test)
    print(sc)

    if sc > best:
        best = sc
        output[sc] = rcv_xgb.best_estimator_

features number = 1


/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Best params: {'alpha': np.float64(5.203592535978188), 'gamma': np.float64(1.2613416206133605), 'learning_rate': np.float64(0.01742818651301447), 'max_depth': 4, 'n_estimators': 11}
Best score: 557.0
1655


In [ ]:
best = 0

output_logreg = {}
# sel = BorutaPy(dtc, n_estimators="auto", verbose=2)
# sel.fit(X.values, Y.values.ravel())

# # sel = SelectFromModel(logreg, prefit=True, max_features=k)

# X_smol = sel.transform(X.values)
X_train, X_test, y_train, y_test = train_test_split(
    X_smol, Y, test_size=0.2
)   

for k in range(1,8):
    print("="*50)
    print(f"features number = {k}")
    dtc = RandomForestClassifier()



    rcv_log = RandomizedSearchCV(
    LogisticRegression(),
    {"C": gamma(2, 2),
    "l1_ratio": beta(2, 2),
    "solver":["saga"],
    "max_iter": [1000]
    },
    n_iter=1000,
    scoring="neg_log_loss",
    n_jobs=-1,
    cv=10)
    
    rcv_log.fit(X_train, y_train.values.ravel())

    print("Best params:", rcv_log.best_params_)
    print("Best score:", rcv_log.best_score_)

    sc = xgb_custom_score(rcv_log.best_estimator_, X_test, y_test)
    print(sc)

    if sc > best:
        best = sc
        output[sc] = rcv_log.best_estimator_

features number = 1
Best params: {'C': np.float64(2.1362490356476487), 'l1_ratio': np.float64(0.12696829559829836), 'max_iter': 1000, 'solver': 'saga'}
Best score: -0.671885414705331
-1235
features number = 2
Best params: {'C': np.float64(2.0677908045730184), 'l1_ratio': np.float64(0.09525047453491155), 'max_iter': 1000, 'solver': 'saga'}
Best score: -0.6718849115426716
-1235
features number = 3
Best params: {'C': np.float64(2.139032595412767), 'l1_ratio': np.float64(0.042544510580462956), 'max_iter': 1000, 'solver': 'saga'}
Best score: -0.6718850255118125
-1235
features number = 4
Best params: {'C': np.float64(2.1979265745233385), 'l1_ratio': np.float64(0.1378869707186234), 'max_iter': 1000, 'solver': 'saga'}
Best score: -0.6718858924920379
-1235
features number = 5


KeyboardInterrupt: 

In [78]:
logreg = rcv_log = RandomizedSearchCV(
    LogisticRegression(),
    {"C": gamma(2, 2),
    "l1_ratio": beta(2, 2),
    "solver":["saga"],
    "max_iter": [1000]
    },
    n_iter=500,
    scoring=xgb_custom_score,
    n_jobs=-1,
    cv=10)

logreg.fit(X_smol, Y.values.ravel())
logreg = logreg.best_estimator_

In [79]:
for k in range(1, 8):
    print("="*50)
    print(f"noVars = {k}")

    sel = SelectFromModel(logreg, prefit=True, max_features=k)

    Xnew = sel.transform(X_smol)

    X_train, X_test, y_train, y_test = train_test_split(
        Xnew, Y, test_size=0.2
    )   

    rcv_log = RandomizedSearchCV(
    LogisticRegression(),
    {"C": gamma(2, 2),
    "l1_ratio": beta(2, 2),
    "solver":["saga"],
    "max_iter": [1000]
    },
    n_iter=1000,
    scoring=xgb_custom_score,
    n_jobs=-1,
    cv=10)

    rcv_log.fit(X_train, y_train.values.ravel())

    print("Best params:", rcv_log.best_params_)
    print("Best score:", rcv_log.best_score_)

    sc = xgb_custom_score(rcv_log.best_estimator_, X_test, y_test)
    print(sc)

    if sc > best:
        best = sc
        output[sc] = rcv_log.best_estimator_

noVars = 1
Best params: {'C': np.float64(4.823786094841828), 'l1_ratio': np.float64(0.08673238811676248), 'max_iter': 1000, 'solver': 'saga'}
Best score: 481.5
970
noVars = 2
Best params: {'C': np.float64(6.935268608268572), 'l1_ratio': np.float64(0.8841342256325436), 'max_iter': 1000, 'solver': 'saga'}
Best score: 252.0
1105
noVars = 3
Best params: {'C': np.float64(2.9995259955587965), 'l1_ratio': np.float64(0.6011846027488237), 'max_iter': 1000, 'solver': 'saga'}
Best score: 114.0
1210
noVars = 4
Best params: {'C': np.float64(2.3871897002568248), 'l1_ratio': np.float64(0.8192647560755264), 'max_iter': 1000, 'solver': 'saga'}
Best score: -66.0
900
noVars = 5
Best params: {'C': np.float64(3.467047180014502), 'l1_ratio': np.float64(0.050961527466273596), 'max_iter': 1000, 'solver': 'saga'}
Best score: -230.5
735
noVars = 6
Best params: {'C': np.float64(2.2705070074605693), 'l1_ratio': np.float64(0.6121068788216494), 'max_iter': 1000, 'solver': 'saga'}
Best score: -298.0
1020
noVars = 7


# Janek

### Setup

In [4]:
def custom_score(estimator, X, y):
    y_probs = estimator.predict_proba(X)[:, 1]
    y_probs_filtered = y_probs[y_probs > 1/3] # based on business score - expected profit
    threshold = np.sort(y_probs_filtered)[::-1][:1000][-1]
    y_pred = (y_probs >= threshold).astype(int)

    y = np.array(y).ravel()

    tp = np.sum((y == 1) & (y_pred == 1))
    fp = np.sum((y == 0) & (y_pred == 1))

    no_variables = X.shape[1]

    return (tp * 10) - (fp * 5) - (no_variables * 200)

In [5]:
def noisy_bootstrap_augment(X, Y, n_samples=None, noise_std=0.01):
    n = n_samples if n_samples is not None else len(X)
    
    idx = np.random.choice(X.index, size=n, replace=True)
    
    X_sampled = X.loc[idx].copy()
    Y_sampled = Y[idx].copy()
    
    noise = np.random.normal(0, noise_std, size=X_sampled.shape)
    X_sampled += noise
    
    # Independent 5% probability coin flip for each individual row
    flip_mask = np.random.binomial(n=1, p=0.05, size=n).astype(bool)
    
    Y_sampled[flip_mask] = 1 - Y_sampled[flip_mask]
        
    return X_sampled.reset_index(drop=True), Y_sampled

In [6]:
X, y, XfinalTest = load_data()
y = y.to_numpy().ravel()
X, XfinalTest = standarize(X, XfinalTest)
X, XfinalTest = variance_threshold(X, XfinalTest, 0.02)
X, XfinalTest = correlation_filter(X, XfinalTest, 0.75)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_estimate, y_estimate = noisy_bootstrap_augment(X_test.reset_index(drop=True), y_test, n_samples=5000)

### ExtraTrees + RandomForest gridsearch

In [61]:
best_score = -np.inf
best_features = None
best_model = None

# best score achieved for k=1
for k in range(1, 11):
    print("="*50)
    print(f"features number = {k}")
    
    sel = SelectFromModel(ExtraTreesClassifier(), max_features=k)
    X_reduced = sel.fit_transform(X_train, y_train)
    X_reduced_test = sel.transform(X_test)

    gridsearch = GridSearchCV(
        RandomForestClassifier(),
        {
            "n_estimators": [50, 100],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2],
            "max_features": ["sqrt", "log2"],
            "criterion": ["gini"]
        },
        scoring=custom_score,
        cv=5,
        n_jobs=-1
    )

    gridsearch.fit(X_reduced, y_train)

    max_score_train = 10*min(y_train.sum(), 1000) - 200*k
    max_score_test = 10*min(y_test.sum(), 1000) - 200*k

    print("Best params:", gridsearch.best_params_)
    print("Best score:", gridsearch.best_score_, "max possible:", max_score_train)

    score = custom_score(gridsearch.best_estimator_, X_reduced_test, y_test)
    print("Test score:", score, "max possible:", max_score_test)
    
    acc_train = accuracy_score(y_train, gridsearch.best_estimator_.predict(X_reduced))
    acc_test = accuracy_score(y_test, gridsearch.best_estimator_.predict(X_reduced_test))
    print("Accuracy:", acc_train, "|", acc_test)

    if score > best_score:
        best_score = score
        best_features = sel.get_feature_names_out()
        best_model = gridsearch.best_estimator_
        

features number = 1
Best params: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 50}
Best score: 1788.0 max possible: 9800
Test score: 2180 max possible: 4720
Accuracy: 0.59025 | 0.509
features number = 2
Best params: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Best score: 1588.0 max possible: 9600
Test score: 1980 max possible: 4520
Accuracy: 0.60425 | 0.588
features number = 3


KeyboardInterrupt: 

### XGBoost + XGBoost randomsearch

In [64]:
best_score = -np.inf
best_features = None
best_model = None

for k in range(1, 11):
    print("="*50)
    print(f"features number = {k}")
    
    sel = SelectFromModel(XGBClassifier(), max_features=k)
    X_reduced = sel.fit_transform(X_train, y_train)
    X_reduced_test = sel.transform(X_test)

    rcv_xgb = RandomizedSearchCV(
        XGBClassifier(),
        {
            "alpha": expon(scale=1),
            "gamma": expon(scale=1),
            "learning_rate": uniform(0.01, 1.0),
            "n_estimators": randint(1, 400),
            "max_depth": randint(1, 10)
        },
        n_iter=100,
        scoring=custom_score,
        n_jobs=-1,
        cv=5
    )

    rcv_xgb.fit(X_reduced, y_train)

    max_score_train = 10*min(y_train.sum(), 1000) - 200*k
    max_score_test = 10*min(y_test.sum(), 1000) - 200*k

    print("Best params:", rcv_xgb.best_params_)
    print("Best score:", rcv_xgb.best_score_, "max possible:", max_score_train)

    score = custom_score(rcv_xgb.best_estimator_, X_reduced_test, y_test)
    print("Test score:", score, "max possible:", max_score_test)
    
    acc_train = accuracy_score(y_train, rcv_xgb.best_estimator_.predict(X_reduced))
    acc_test = accuracy_score(y_test, rcv_xgb.best_estimator_.predict(X_reduced_test))
    print("Accuracy:", acc_train, "|", acc_test)

    if score > best_score:
        best_score = score
        best_features = sel.get_feature_names_out()
        best_model = rcv_xgb.best_estimator_
        

features number = 1
Best params: {'alpha': np.float64(1.1888138750655821), 'gamma': np.float64(0.6222443451816919), 'learning_rate': np.float64(0.762509827234386), 'max_depth': 9, 'n_estimators': 307}
Best score: 1791.0 max possible: 9800
Test score: 2180 max possible: 4720
Accuracy: 0.56775 | 0.545
features number = 2
Best params: {'alpha': np.float64(0.7419388688484271), 'gamma': np.float64(2.3037929149748493), 'learning_rate': np.float64(0.12021297186919931), 'max_depth': 2, 'n_estimators': 390}
Best score: 1588.0 max possible: 9600
Test score: 1980 max possible: 4520
Accuracy: 0.60175 | 0.586
features number = 3


Exception ignored while calling ctypes callback function <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x7dbc319cbee0>>:
Traceback (most recent call last):
  File "/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/xgboost/core.py", line 662, in _next_wrapper
    return self._handle_exception(lambda: int(self.next(input_data)), 0)
  File "/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/xgboost/core.py", line 575, in _handle_exception
    return fn()
  File "/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/xgboost/core.py", line 662, in <lambda>
    return self._handle_exception(lambda: int(self.next(input_data)), 0)
  File "/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/xgboost/data.py", line 1642, in next
    input_data(**self.kwargs)
  File "/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kw

KeyboardInterrupt: 

In [ ]:
X_best = X[best_features]
X_best_train = X_train[best_features]
X_best_test = X_test[best_features]
print(custom_score(best_model, X_best, y))
print(custom_score(best_model, X_best_train, y_train))
print(custom_score(best_model, X_best_test, y_test))

4605
4215
2180


### ExtraTrees + XGBoost randomsearch

In [34]:
best_score = -np.inf
best_features = None
best_model = None

for k in range(1, 11):
    print("="*50)
    print(f"features number = {k}")
    
    sel = SelectFromModel(ExtraTreesClassifier(), max_features=k)
    X_reduced = sel.fit_transform(X_train, y_train)
    X_reduced_test = sel.transform(X_test)
    X_reduced_estimate = sel.transform(X_estimate)

    rcv_xgb = RandomizedSearchCV(
        XGBClassifier(eval_metric='logloss', random_state=42),
        {
            "n_estimators": randint(100, 400),
            "max_depth": randint(1, 4),
            "learning_rate": uniform(0.01, 0.2),
            "subsample": uniform(0.6, 0.4),
            "reg_lambda": uniform(0, 2),
        },
        n_iter=100,
        scoring="neg_log_loss",
        n_jobs=-1,
        cv=5,
        random_state=42
    )

    rcv_xgb.fit(X_reduced, y_train)

    max_score_train = 10*min(y_train.sum(), 1000) - 200*k
    max_score_test = 10*min(y_test.sum(), 1000) - 200*k
    max_score_estimate = 10*min(y_estimate.sum(), 1000) - 200*k

    print("Best params:", rcv_xgb.best_params_)
    print("Best CV score:", custom_score(rcv_xgb.best_estimator_, X_reduced, y_train), "max possible:", max_score_train)
    print("Test score:", custom_score(rcv_xgb.best_estimator_, X_reduced_test, y_test), "max possible:", max_score_test)

    score = custom_score(rcv_xgb.best_estimator_, X_reduced_estimate, y_estimate)
    print("Estimated final score:", score, "max possible", max_score_estimate)
    
    acc_train = accuracy_score(y_train, rcv_xgb.best_estimator_.predict(X_reduced))
    acc_test = accuracy_score(y_test, rcv_xgb.best_estimator_.predict(X_reduced_test))
    print("Accuracy:", acc_train, "|", acc_test)

    if score > best_score:
        best_score = score
        best_features = sel.get_feature_names_out()
        best_model = rcv_xgb.best_estimator_

features number = 1
Best params: {'learning_rate': np.float64(0.01872075435088675), 'max_depth': 2, 'n_estimators': 145, 'reg_lambda': np.float64(1.829728780440897), 'subsample': np.float64(0.7480634801021777)}
Best CV score: 4090 max possible: 9800
Test score: 2180 max possible: 4720
Estimated final score: 4535 max possible 9800
Accuracy: 0.5595 | 0.553
features number = 2
Best params: {'learning_rate': np.float64(0.013317565785571231), 'max_depth': 3, 'n_estimators': 246, 'reg_lambda': np.float64(0.6973319745834587), 'subsample': np.float64(0.6384706204365683)}
Best CV score: 5175 max possible: 9600
Test score: 1980 max possible: 4520
Estimated final score: 4725 max possible 9600
Accuracy: 0.61575 | 0.584
features number = 3
Best params: {'learning_rate': np.float64(0.013317565785571231), 'max_depth': 3, 'n_estimators': 246, 'reg_lambda': np.float64(0.6973319745834587), 'subsample': np.float64(0.6384706204365683)}
Best CV score: 5635 max possible: 9400
Test score: 1780 max possible: 

In [35]:
X_best = X[best_features]
X_best_train = X_train[best_features]
X_best_test = X_test[best_features]
X_best_estimate = X_estimate[best_features]
print("Total | Train | Test | Estimated")
print(f"{custom_score(best_model, X_best, y)} | {custom_score(best_model, X_best_train, y_train)} | {custom_score(best_model, X_best_test, y_test)} | {custom_score(best_model, X_best_estimate, y_estimate)}")

Total | Train | Test | Estimated
5915 | 5810 | 1550 | 5080


### ExtraTrees + XGBoost randomsearch (get_data)

In [39]:
X, y, XfinalTest = load_data()
Xnowe, top20 = get_data()
X = Xnowe
XfinalTest = XfinalTest[top20]

y = y.to_numpy().ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_estimate, y_estimate = noisy_bootstrap_augment(X_test.reset_index(drop=True), y_test, n_samples=5000)

--- STARTING STRICT PIPELINE ---
Initial features: 500
Stage 1 (Variance & Collinearity) survivors: 485
Stage 2 (L1 Penalty) survivors: 280
check
Stage 3 (RFECV) survivors: 16
Stage 4 (Shadow Feature Protocol) survivors: 10
Stage 5 (OOF Permutation) Final Selection: 10 features.
--- PIPELINE COMPLETE ---



In [40]:
best_score = -np.inf
best_features = None
best_model = None

for k in range(1, X.shape[1] + 1):
    print("="*50)
    print(f"features number = {k}")
    
    sel = SelectFromModel(ExtraTreesClassifier(), max_features=k)
    X_reduced = sel.fit_transform(X_train, y_train)
    X_reduced_test = sel.transform(X_test)
    X_reduced_estimate = sel.transform(X_estimate)

    rcv_xgb = RandomizedSearchCV(
        XGBClassifier(eval_metric='logloss', random_state=42),
        {
            "n_estimators": randint(100, 400),
            "max_depth": randint(1, 4),
            "learning_rate": uniform(0.01, 0.2),
            "subsample": uniform(0.6, 0.4),
            "reg_lambda": uniform(0, 2),
        },
        n_iter=100,
        scoring="neg_log_loss",
        n_jobs=-1,
        cv=5,
        random_state=42
    )

    rcv_xgb.fit(X_reduced, y_train)

    max_score_train = 10*min(y_train.sum(), 1000) - 200*k
    max_score_test = 10*min(y_test.sum(), 1000) - 200*k
    max_score_estimate = 10*min(y_estimate.sum(), 1000) - 200*k

    print("Best params:", rcv_xgb.best_params_)
    print("Best CV score:", custom_score(rcv_xgb.best_estimator_, X_reduced, y_train), "max possible:", max_score_train)
    print("Test score:", custom_score(rcv_xgb.best_estimator_, X_reduced_test, y_test), "max possible:", max_score_test)

    score = custom_score(rcv_xgb.best_estimator_, X_reduced_estimate, y_estimate)
    print("Estimated final score:", score, "max possible", max_score_estimate)
    
    acc_train = accuracy_score(y_train, rcv_xgb.best_estimator_.predict(X_reduced))
    acc_test = accuracy_score(y_test, rcv_xgb.best_estimator_.predict(X_reduced_test))
    print("Accuracy:", acc_train, "|", acc_test)

    if score > best_score:
        best_score = score
        best_features = sel.get_feature_names_out()
        best_model = rcv_xgb.best_estimator_

features number = 1
Best params: {'learning_rate': np.float64(0.02029575024999787), 'max_depth': 1, 'n_estimators': 323, 'reg_lambda': np.float64(1.8165317719333074), 'subsample': np.float64(0.695824756266789)}
Best CV score: 4395 max possible: 9800
Test score: 2180 max possible: 4720
Estimated final score: 3845 max possible 9800
Accuracy: 0.56425 | 0.545
features number = 2
Best params: {'learning_rate': np.float64(0.15264895744459903), 'max_depth': 1, 'n_estimators': 104, 'reg_lambda': np.float64(1.1225543951389925), 'subsample': np.float64(0.9083868719818244)}
Best CV score: 4735 max possible: 9600
Test score: 1980 max possible: 4520
Estimated final score: 3745 max possible 9600
Accuracy: 0.60175 | 0.571
features number = 3
Best params: {'learning_rate': np.float64(0.07092275383467414), 'max_depth': 1, 'n_estimators': 191, 'reg_lambda': np.float64(0.8803049874792026), 'subsample': np.float64(0.6488152939379115)}
Best CV score: 5305 max possible: 9400
Test score: 1780 max possible: 4

In [41]:
X_best = X[best_features]
X_best_train = X_train[best_features]
X_best_test = X_test[best_features]
X_best_estimate = X_estimate[best_features]
print("Total | Train | Test | Estimated")
print(f"{custom_score(best_model, X_best, y)} | {custom_score(best_model, X_best_train, y_train)} | {custom_score(best_model, X_best_test, y_test)} | {custom_score(best_model, X_best_estimate, y_estimate)}")

Total | Train | Test | Estimated
5445 | 5190 | 1410 | 4505


In [ ]:
best_score = -np.inf
best_features = None
best_model = None

for k in range(1, X.shape[1] + 1):
    print("="*50)
    print(f"features number = {k}")
    
    X_reduced = X_train.iloc[:, :k]
    X_reduced_test = X_test.iloc[:, :k]
    X_reduced_estimate = X_estimate.iloc[:, :k]

    rcv_xgb = RandomizedSearchCV(
        XGBClassifier(eval_metric='logloss', random_state=42),
        {
            "n_estimators": randint(100, 400),
            "max_depth": randint(1, 4),
            "learning_rate": uniform(0.01, 0.2),
            "subsample": uniform(0.6, 0.4),
            "reg_lambda": uniform(0, 2),
        },
        n_iter=100,
        scoring="neg_log_loss",
        n_jobs=-1,
        cv=5,
        random_state=42
    )

    rcv_xgb.fit(X_reduced, y_train)

    max_score_train = 10*min(y_train.sum(), 1000) - 200*k
    max_score_test = 10*min(y_test.sum(), 1000) - 200*k
    max_score_estimate = 10*min(y_estimate.sum(), 1000) - 200*k

    print("Best params:", rcv_xgb.best_params_)
    print("Best CV score:", custom_score(rcv_xgb.best_estimator_, X_reduced, y_train), "max possible:", max_score_train)
    print("Test score:", custom_score(rcv_xgb.best_estimator_, X_reduced_test, y_test), "max possible:", max_score_test)

    score = custom_score(rcv_xgb.best_estimator_, X_reduced_estimate, y_estimate)
    print("Estimated final score:", score, "max possible", max_score_estimate)
    
    acc_train = accuracy_score(y_train, rcv_xgb.best_estimator_.predict(X_reduced))
    acc_test = accuracy_score(y_test, rcv_xgb.best_estimator_.predict(X_reduced_test))
    print("Accuracy:", acc_train, "|", acc_test)

    if score > best_score:
        best_score = score
        best_features = X_train.columns[:k]
        best_model = rcv_xgb.best_estimator_

features number = 1
Best params: {'learning_rate': np.float64(0.02029575024999787), 'max_depth': 1, 'n_estimators': 323, 'reg_lambda': np.float64(1.8165317719333074), 'subsample': np.float64(0.695824756266789)}
Best CV score: 4395 max possible: 9800
Test score: 2180 max possible: 4720
Estimated final score: 3845 max possible 9800
Accuracy: 0.56425 | 0.545
features number = 2
Best params: {'learning_rate': np.float64(0.05316420549936864), 'max_depth': 1, 'n_estimators': 202, 'reg_lambda': np.float64(0.170694929987536), 'subsample': np.float64(0.6206726884674431)}
Best CV score: 4815 max possible: 9600
Test score: 1980 max possible: 4520
Estimated final score: 4155 max possible 9600
Accuracy: 0.594 | 0.601
features number = 3
Best params: {'learning_rate': np.float64(0.07092275383467414), 'max_depth': 1, 'n_estimators': 191, 'reg_lambda': np.float64(0.8803049874792026), 'subsample': np.float64(0.6488152939379115)}
Best CV score: 5305 max possible: 9400
Test score: 1780 max possible: 4320

In [53]:
X_best = X[best_features]
X_best_train = X_train[best_features]
X_best_test = X_test[best_features]
X_best_estimate = X_estimate[best_features]
print("Total | Train | Test | Estimated")
print(f"{custom_score(best_model, X_best, y)} | {custom_score(best_model, X_best_train, y_train)} | {custom_score(best_model, X_best_test, y_test)} | {custom_score(best_model, X_best_estimate, y_estimate)}")

Total | Train | Test | Estimated
5345 | 5180 | 1595 | 4205


### ExtraTrees + XGBoost randomsearch (give_data)

In [7]:
X, y, XfinalTest = load_data()
Xnowe, top20 = give_data()
X = Xnowe
XfinalTest = XfinalTest[top20]

y = y.to_numpy().ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_estimate, y_estimate = noisy_bootstrap_augment(X_test.reset_index(drop=True), y_test, n_samples=5000)

Original dataset shape: (5000, 500)
Features after dropping highly collinear ones: 489
Features passing L1 strict penalty: 280


/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Features passing RFECV: 19

Features surviving BOTH strict tests: 13

--- FINAL STRICT TOP 20 FEATURES ---


,Feature,Importance_Mean,Importance_Std
11,V345,0.063308,0.001899
7,V342,0.062351,0.001837
12,V380,0.060629,0.002938
0,V199,0.058682,0.001722
5,V329,0.046936,0.002165
9,V390,0.045761,0.001402
10,V32,0.043058,0.002138
6,V121,0.041547,0.000897
8,V262,0.040224,0.001749
3,V483,0.039923,0.001976



Final strict dataset shape: (5000, 13)


In [8]:
best_score = -np.inf
best_features = None
best_model = None

for k in range(1, X.shape[1] + 1):
    print("="*50)
    print(f"features number = {k}")
    
    sel = SelectFromModel(ExtraTreesClassifier(), max_features=k)
    X_reduced = sel.fit_transform(X_train, y_train)
    X_reduced_test = sel.transform(X_test)
    X_reduced_estimate = sel.transform(X_estimate)

    rcv_xgb = RandomizedSearchCV(
        XGBClassifier(eval_metric='logloss', random_state=42),
        {
            "n_estimators": randint(100, 400),
            "max_depth": randint(1, 4),
            "learning_rate": uniform(0.01, 0.2),
            "subsample": uniform(0.6, 0.4),
            "reg_lambda": uniform(0, 2),
        },
        n_iter=100,
        scoring="neg_log_loss",
        n_jobs=-1,
        cv=5,
        random_state=42
    )

    rcv_xgb.fit(X_reduced, y_train)

    max_score_train = 10*min(y_train.sum(), 1000) - 200*k
    max_score_test = 10*min(y_test.sum(), 1000) - 200*k
    max_score_estimate = 10*min(y_estimate.sum(), 1000) - 200*k

    print("Best params:", rcv_xgb.best_params_)
    print("Best CV score:", custom_score(rcv_xgb.best_estimator_, X_reduced, y_train), "max possible:", max_score_train)
    print("Test score:", custom_score(rcv_xgb.best_estimator_, X_reduced_test, y_test), "max possible:", max_score_test)

    score = custom_score(rcv_xgb.best_estimator_, X_reduced_estimate, y_estimate)
    print("Estimated final score:", score, "max possible", max_score_estimate)
    
    acc_train = accuracy_score(y_train, rcv_xgb.best_estimator_.predict(X_reduced))
    acc_test = accuracy_score(y_test, rcv_xgb.best_estimator_.predict(X_reduced_test))
    print("Accuracy:", acc_train, "|", acc_test)

    if score > best_score:
        best_score = score
        best_features = sel.get_feature_names_out()
        best_model = rcv_xgb.best_estimator_

features number = 1
Best params: {'learning_rate': np.float64(0.02569127626845319), 'max_depth': 1, 'n_estimators': 216, 'reg_lambda': np.float64(0.16174593323439534), 'subsample': np.float64(0.7713257899760431)}
Best CV score: 3950 max possible: 9800
Test score: 2180 max possible: 4720
Estimated final score: 3360 max possible 9800
Accuracy: 0.5565 | 0.535
features number = 2
Best params: {'learning_rate': np.float64(0.05316420549936864), 'max_depth': 1, 'n_estimators': 202, 'reg_lambda': np.float64(0.170694929987536), 'subsample': np.float64(0.6206726884674431)}
Best CV score: 4465 max possible: 9600
Test score: 1980 max possible: 4520
Estimated final score: 4155 max possible 9600
Accuracy: 0.58325 | 0.578
features number = 3
Best params: {'learning_rate': np.float64(0.07092275383467414), 'max_depth': 1, 'n_estimators': 191, 'reg_lambda': np.float64(0.8803049874792026), 'subsample': np.float64(0.6488152939379115)}
Best CV score: 5125 max possible: 9400
Test score: 1780 max possible: 4

In [9]:
X_best = X[best_features]
X_best_train = X_train[best_features]
X_best_test = X_test[best_features]
X_best_estimate = X_estimate[best_features]
print("Total | Train | Test | Estimated")
print(f"{custom_score(best_model, X_best, y)} | {custom_score(best_model, X_best_train, y_train)} | {custom_score(best_model, X_best_test, y_test)} | {custom_score(best_model, X_best_estimate, y_estimate)}")

Total | Train | Test | Estimated
4750 | 4465 | 1980 | 4155


# Gemini

In [215]:
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from scipy.stats import loguniform, uniform, randint
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix

def custom_profit_score(y_true, y_pred, no_vars):
    """
    Calculates the financial score based on project constraints.
    y_true: actual labels (0 or 1)
    y_pred: predicted labels (0 or 1)
    no_vars: number of variables used in the model
    """
    # Handle edge case where no positive predictions are made
    if np.sum(y_pred) == 0:
        return -(no_vars * 200)
        
    # Calculate True Positives and False Positives
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    
    # Apply the financial formula
    score = (tp * 10) - (fp * 5) - (no_vars * 200)
    
    return score

In [216]:
def tune_logistic_regression(X, y):
    """Tunes a Logistic Regression model with L1 (Lasso) penalty."""
    model = LogisticRegression(l1_ratio=1, solver='liblinear', max_iter=1000)
    
    param_dist = {
        'C': loguniform(1e-4, 10) # Search across a wide range of regularization strengths
    }
    
    # We use standard accuracy or roc_auc for internal tuning to find a stable model, 
    # but the stepwise loop (below) will use the custom profit score to select features.
    search = RandomizedSearchCV(model, param_distributions=param_dist, 
                                n_iter=10, cv=3, scoring='roc_auc', 
                                random_state=42, n_jobs=-1)
    search.fit(X, y)
    return search.best_estimator_

def tune_xgboost(X, y):
    """Tunes a shallow XGBoost model to prevent overfitting."""
    model = XGBClassifier(eval_metric='logloss')
    
    param_dist = {
        'max_depth': randint(2, 5), # Keep trees shallow
        'learning_rate': loguniform(0.01, 0.2),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4)
    }
    
    search = RandomizedSearchCV(model, param_distributions=param_dist, 
                                n_iter=10, cv=3, scoring='roc_auc', 
                                random_state=42, n_jobs=-1)
    search.fit(X, y)
    return search.best_estimator_

In [ ]:
def forward_stepwise_selection(X_train, y_train, X_val, y_val, model_type='logreg'):
    """
    Performs forward stepwise selection maximizing the custom profit score.
    model_type: 'logreg' or 'xgboost'
    """
    initial_features = list(X_train.columns)
    best_features = []
    best_overall_score = float('-inf')
    
    # Threshold for predicting a positive class (break-even point is >33.3%)
    probability_threshold = 0.28
    
    print(f"Starting Forward Selection for {model_type.upper()}...")
    
    while len(initial_features) > 0:
        scores_with_new_feature = []
        
        # Test adding each available feature one by one
        for feature in initial_features:
            current_features_to_test = best_features + [feature]
            num_vars = len(current_features_to_test)
            
            # Subset the data
            X_train_sub = X_train[current_features_to_test]
            X_val_sub = X_val[current_features_to_test]
            
            # Tune and train the model on this specific feature subset
            if model_type == 'logreg':
                model = tune_logistic_regression(X_train_sub, y_train)
            else:
                model = tune_xgboost(X_train_sub, y_train)
            
            # Predict probabilities on the validation set
            val_probs = model.predict_proba(X_val_sub)[:, 1]
            
            # Convert probabilities to binary predictions using our break-even threshold
            if (val_probs >= probability_threshold).sum() > 1000:
                best_indices = np.argsort(val_probs)[::-1][:1000]
                val_preds = np.zeros_like(val_probs)
                val_preds[best_indices] = 1
            else:
                val_preds = (val_probs >= probability_threshold).astype(int)
            
            # Calculate the custom financial score
            score = custom_profit_score(y_val, val_preds, num_vars)
            scores_with_new_feature.append((score, feature))
            
        # Find the feature that provided the highest profit jump in this round
        scores_with_new_feature.sort(reverse=True) # Sort descending by score
        best_round_score, best_round_feature = scores_with_new_feature[0]
        
        # Did adding this feature increase our overall profit?
        if best_round_score > best_overall_score:
            best_overall_score = best_round_score
            best_features.append(best_round_feature)
            initial_features.remove(best_round_feature)
            print(f"Added feature '{best_round_feature}'. New best score: EUR {best_overall_score} (Features: {len(best_features)})")
        else:
            # If the score drops (because the feature didn't overcome the EUR 200 penalty), STOP.
            print(f"Stopping: No remaining features justify the EUR 200 cost.")
            break
            
    return best_features, best_overall_score

In [226]:
import pandas as pd
import numpy as np

def noisy_bootstrap_augment(X, Y, n_samples=None, noise_std=0.01):
    n = n_samples if n_samples is not None else len(X)
    
    idx = np.random.choice(X.index, size=n, replace=True)
    
    X_sampled = X.loc[idx].copy()
    Y_sampled = Y[idx].copy()
    
    noise = np.random.normal(0, noise_std, size=X_sampled.shape)
    X_sampled += noise
    
    # Independent 5% probability coin flip for each individual row
    flip_mask = np.random.binomial(n=1, p=0.05, size=n).astype(bool)
    
    Y_sampled[flip_mask] = 1 - Y_sampled[flip_mask]
        
    return X_sampled, Y_sampled

In [230]:
import pandas as pd

# Assuming you have loaded your data into a DataFrame 'X' and Series 'y'
# X = pd.read_csv('x_train.txt', sep='\t') 
# y = pd.read_csv('y_train.txt', sep='\t')

X_sampled, y_sampled = noisy_bootstrap_augment(X, y, 15_000, 0.02)

# 1. Split data (Crucial for honest profit evaluation)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_sampled, y_sampled, test_size=0.3, stratify=y_sampled, random_state=42
)

# 2. Run Stepwise Selection for Logistic Regression
print("--- LOGISTIC REGRESSION ---")
best_logreg_vars, logreg_profit = forward_stepwise_selection(
    X_train_split, y_train_split, X_val_split, y_val_split, model_type='logreg'
)

# 3. Run Stepwise Selection for XGBoost
print("\n--- XGBOOST ---")
best_xgb_vars, xgb_profit = forward_stepwise_selection(
    X_train_split, y_train_split, X_val_split, y_val_split, model_type='xgboost'
)

print("\n--- FINAL RESULTS ---")
print(f"LogReg Best Variables: {best_logreg_vars} | Val Profit: EUR {logreg_profit}")
print(f"XGBoost Best Variables: {best_xgb_vars} | Val Profit: EUR {xgb_profit}")

--- LOGISTIC REGRESSION ---
Starting Forward Selection for LOGREG...
Added feature 'V99'. New best score: EUR 11245 (Features: 1)
Stopping: No remaining features justify the EUR 200 cost.

--- XGBOOST ---
Starting Forward Selection for XGBOOST...
Added feature 'V273'. New best score: EUR 11335 (Features: 1)
Stopping: No remaining features justify the EUR 200 cost.

--- FINAL RESULTS ---
LogReg Best Variables: ['V99'] | Val Profit: EUR 11245
XGBoost Best Variables: ['V273'] | Val Profit: EUR 11335


In [231]:
def evaluate_model_metrics(model, X_val, y_val, no_vars, threshold=0.334):
    """
    Evaluates standard metrics and profit based on a custom probability threshold.
    """
    # 1. Get probabilities for the positive class (class 1)
    val_probs = model.predict_proba(X_val)[:, 1]
    
    # 2. Apply the profit-maximizing threshold
    val_preds = (val_probs >= threshold).astype(int)
    
    # 3. Calculate standard metrics
    accuracy = accuracy_score(y_val, val_preds)
    
    # zero_division=0 prevents errors if the model makes absolutely no positive predictions
    precision = precision_score(y_val, val_preds, zero_division=0) 
    
    # 4. Extract confusion matrix components to calculate exact profit
    tn, fp, fn, tp = confusion_matrix(y_val, val_preds, labels=[0, 1]).ravel()
    profit = (tp * 10) - (fp * 5) - (no_vars * 200)
    
    # 5. Print a clean summary report
    print(f"--- Model Evaluation (Threshold: {threshold}) ---")
    print(f"Variables Used:  {no_vars}")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Precision:       {precision:.4f}")
    print(f"True Positives:  {tp}")
    print(f"False Positives: {fp}")
    print(f"Net Profit:      EUR {profit}")
    print("-" * 40)
    
    return accuracy, precision, profit

In [232]:
# --- Assuming `best_logreg_vars` was found by the stepwise function ---

print("\nEvaluating Final Logistic Regression Model...")

# Subset the training and validation data using ONLY the selected best variables
X_train_final = X_train_split[best_logreg_vars]
X_val_final = X_val_split[best_logreg_vars]
num_final_vars = len(best_logreg_vars)

# Retrain the optimized model on just the selected variables
final_logreg_model = tune_logistic_regression(X_train_final, y_train_split)

# Evaluate metrics using the custom function
acc, prec, prof = evaluate_model_metrics(
    model=final_logreg_model, 
    X_val=X_val_final, 
    y_val=y_val_split, 
    no_vars=num_final_vars, 
    threshold=0.28
)

# --- Assuming `best_xgb_vars` was found by the stepwise function ---

print("\nEvaluating Final XGBoost Model...")

# Subset the training and validation data using ONLY the selected best variables
X_train_final = X_train_split[best_xgb_vars]
X_val_final = X_val_split[best_xgb_vars]
num_final_vars = len(best_xgb_vars)

# Retrain the optimized model on just the selected variables
final_xgb_model = tune_xgboost(X_train_final, y_train_split)

# Evaluate metrics using the custom function
acc, prec, prof = evaluate_model_metrics(
    model=final_xgb_model, 
    X_val=X_val_final, 
    y_val=y_val_split, 
    no_vars=num_final_vars, 
    threshold=0.28
)


Evaluating Final Logistic Regression Model...
--- Model Evaluation (Threshold: 0.28) ---
Variables Used:  1
Accuracy:        0.5029
Precision:       0.5029
True Positives:  2263
False Positives: 2237
Net Profit:      EUR 11245
----------------------------------------

Evaluating Final XGBoost Model...
--- Model Evaluation (Threshold: 0.28) ---
Variables Used:  1
Accuracy:        0.5069
Precision:       0.5049
True Positives:  2263
False Positives: 2219
Net Profit:      EUR 11335
----------------------------------------
